<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c34-header.png" alt="Nextia Learning · Anomaly Detection: Find What Is Unusual" width="100%">

# Control charts on a real production line

**[Control charts on a real production line](https://learning.nextia-ai.com/courses/anomaly-detection/m10/control-charts-on-a-real-production-line/)** · Anomaly Detection: Find What Is Unusual · Module 10, case study 4 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will** watch a real semiconductor line with control charts and a drift report: set phase I limits, find out how many of its 590 sensors are noise, chart the sensors that moved, and compare the detectors under one weekly inspection budget. The pass/fail results of the line test are only a check: no step learns from them.

| | |
|---|---|
| **Time** | About 2 hours with the case study page; the notebook runs in under a minute |
| **Needs** | An internet connection for the setup cells (2 MB). No account. |
| **You should know** | Phase I and phase II, from [Control charts](https://learning.nextia-ai.com/courses/anomaly-detection/m03/control-charts/); EWMA, from [Small, lasting shifts: CUSUM and EWMA](https://learning.nextia-ai.com/courses/anomaly-detection/m03/small-lasting-shifts-cusum-and-ewma/); the drift report with the false discovery rate, from [Drift in many features](https://learning.nextia-ai.com/courses/anomaly-detection/m03/drift-in-many-features/); the review budget, from [The alert budget](https://learning.nextia-ai.com/courses/anomaly-detection/m07/the-alert-budget/). |
| **Data** | UCI SECOM, by Michael McCann and Adrian Johnston, CC BY 4.0, [doi.org/10.24432/C54305](https://doi.org/10.24432/C54305) |
| **Tested** | Python 3.12.13 in Jupyter on macOS (Apple silicon), from a fresh start (*Restart and run all*), with the package versions of the setup cell |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/anomaly-detection/m10/control-charts-on-a-real-production-line/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C34/notebooks/M10-L04-control-charts-on-a-real-production-line/control-charts-on-a-real-production-line-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: pandas 3.1.1, numpy 2.5.4, scipy 1.18.1, scikit-learn 1.9.1, matplotlib 3.11.2. It needs an internet connection. The first time, it takes up to a minute.

You should see one line per package, with the same versions. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 numpy==2.5.4 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2
from importlib.metadata import version
for p in ['pandas', 'numpy', 'scipy', 'scikit-learn', 'matplotlib']:
    print(p, version(p))

## Setup: get the data

Run the next cell. It downloads `secom.zip` (2 MB) from the UCI Machine Learning Repository into a folder `secom-case-study`, checks its SHA-256 checksum and unzips it. If the UCI link does not work, it uses the unchanged copy in the course's labs repository. It needs no account.

You should see `Ready:` and two files with their sizes. If you see `Could not get secom.zip`, check your internet connection (in Kaggle, turn on **Internet** in the notebook settings), delete the folder `secom-case-study`, and run the cell again.

The data is "SECOM" by Michael McCann and Adrian Johnston, UCI Machine Learning Repository, [doi.org/10.24432/C54305](https://doi.org/10.24432/C54305), licensed under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). The [dataset card](dataset.md) has the details.

In [ ]:
# Setup: download SECOM from the UCI repository into secom-case-study/ and check it. Standard library only.
import hashlib, os, shutil, subprocess, urllib.request, zipfile
from pathlib import Path

URL = "https://archive.ics.uci.edu/static/public/179/secom.zip"
BACKUP = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C34/notebooks/M10-L04-control-charts-on-a-real-production-line/data/secom.zip"
SHA256 = "eea568baf3c2229096d7d294cf0b096b5502bd96d92c0b80a65b84714059be8e"
HERE = Path.cwd()
# The notebook works in its own folder secom-case-study/, marked with a file, so it never writes into another project.
if Path(".nextia-notebook").exists():          # the cell ran before: start again from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
WORK = Path("secom-case-study").resolve()
WORK.mkdir(exist_ok=True)
(WORK / ".nextia-notebook").touch()
zip_path = WORK / "secom.zip"

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        subprocess.run(["curl", "-fsSL", "-o", str(path), url])

if not zip_path.exists() or sha256(zip_path) != SHA256:
    for copy in (HERE / "secom.zip", HERE / "data" / "secom.zip"):   # a copy next to the notebook
        if copy.exists():
            shutil.copy(copy, zip_path)
            break
    else:
        download(URL, zip_path)
        if not zip_path.exists() or sha256(zip_path) != SHA256:
            download(BACKUP, zip_path)                                  # the unchanged copy in the course's labs
if not zip_path.exists() or sha256(zip_path) != SHA256:
    raise SystemExit("Could not get secom.zip with the right checksum. Check your internet connection, "
                     "delete the folder secom-case-study, and run this cell again.")
with zipfile.ZipFile(zip_path) as z:
    z.extractall(WORK)
os.chdir(WORK)
print("Ready:", ", ".join(f"{n} ({Path(n).stat().st_size / 1e6:.2f} MB)" for n in ("secom.data", "secom_labels.data")))

## Setup: helpers

The next cell defines the check helper `expect()`, which says "too high" or "too low" and never shows the answer. It also defines three small functions from the course: `bh()` (the false discovery rate), `psi()` and `ewma()`. Run the cell. It prints nothing.

In [ ]:
# Helpers: the checks of your answers, the chart style, and three small functions from the course.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.ensemble import IsolationForest
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)
warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams.update({"figure.figsize": (9, 3.2), "axes.spines.top": False, "axes.spines.right": False})

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number with the expected one. It says "too high" or "too low", never the answer."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif abs(yours - expected) <= tolerance * max(1, abs(expected)):
        print(f"Check passed: {what} is {yours}.")
    else:
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")

def bh(p, q=0.05):
    """Benjamini-Hochberg: True for each test that the false discovery rate q keeps."""
    p = np.asarray(p, float)
    order, m = np.argsort(p), len(p)
    under = p[order] <= q * np.arange(1, m + 1) / m
    keep = np.zeros(m, bool)
    if under.any():
        keep[order[: np.nonzero(under)[0].max() + 1]] = True
    return keep

def psi(reference, current, bins=10):
    """Population stability index, with 10 bins from the reference's deciles."""
    reference, current = reference.dropna(), current.dropna()
    edges = np.unique(np.quantile(reference, np.linspace(0, 1, bins + 1)))
    if len(edges) < 3:
        return np.nan
    edges[0], edges[-1] = -np.inf, np.inf
    a = np.clip(np.histogram(reference, edges)[0] / len(reference), 1e-4, None)
    b = np.clip(np.histogram(current, edges)[0] / len(current), 1e-4, None)
    return float(((b - a) * np.log(b / a)).sum())

def ewma(values, lam=0.1):
    """The EWMA of a series: lam of the new value, 1 - lam of the old average. It starts at 0."""
    e, out = 0.0, []
    for v in values:
        e = lam * v + (1 - lam) * e
        out.append(e)
    return np.array(out)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The data and the first checks

One row is one **unit** of production with 590 sensor readings. A second file has the result of the line test at the end (pass or fail) and the time of the test. Run the next cell.

> **Check.** You should see `(1567, 590)`, 104 fails (6.6%), and `in time order: True`.

In [ ]:
sensors = pd.read_csv("secom.data", sep=r"\s+", header=None)
sensors.columns = [f"s{i:03d}" for i in range(sensors.shape[1])]      # the sensors have numbers, not names
labels = pd.read_csv("secom_labels.data", sep=" ", header=None, names=["result", "time"])
labels["time"] = pd.to_datetime(labels["time"], format="%d/%m/%Y %H:%M:%S")
fail = (labels["result"] == 1).to_numpy()                            # 1 = failed the line test, -1 = passed
print(sensors.shape, f"| fails: {fail.sum()} ({fail.mean():.1%})")
print("from", labels.time.min(), "to", labels.time.max(), "| in time order:", labels.time.is_monotonic_increasing)
print(f"missing readings: {sensors.isna().mean().mean():.1%} | units a day: median "
      f"{labels.groupby(labels.time.dt.date).size().median():.0f}, most {labels.groupby(labels.time.dt.date).size().max()}")

We split the 13 weeks in time. **Phase I** sets the limits. The **first week** after it is the first look. The **comparison** weeks compare the detectors. The **new week** is kept for the end-to-end run.

> **Check.** You should see 618 units in phase I and 722 in the comparison weeks. The fail rate is 10.5% in phase I and 5.0% in the comparison weeks.

In [ ]:
phase1 = (labels.time < "2008-09-01").to_numpy()                                       # phase I: the limits
first_week = ((labels.time >= "2008-09-01") & (labels.time < "2008-09-08")).to_numpy()   # the first look
compare_weeks = ((labels.time >= "2008-09-08") & (labels.time < "2008-10-13")).to_numpy()  # the comparison
new_week = (labels.time >= "2008-10-13").to_numpy()                                    # new input, at the end
week = labels.time.dt.to_period("W").dt.start_time.dt.strftime("%Y-%m-%d").to_numpy()  # the Monday of each week
for name, m in [("phase I", phase1), ("first week", first_week), ("comparison", compare_weeks), ("new week", new_week)]:
    t = labels.time[m]
    print(f"{name:11} {t.min():%d %b} to {t.max():%d %b}: {m.sum():4} units, {fail[m].sum():3} fails ({fail[m].mean():.1%})")

## 2. How many sensors are noise?

Some sensors cannot tell us anything: they never change, or most of their readings are missing. Run the next cell.

> **Check.** You should see 116 constant, 28 more than half empty, 14 stuck on one value in phase I, and 432 kept.

A **stuck** sensor read the same value on more than 95% of the phase I units. Its moving range is close to 0, so its limits are close to 0 wide, and every later change looks like a huge alarm.

In [ ]:
constant = sensors.nunique() <= 1                                        # the same value on every unit
mostly_empty = sensors.isna().mean() > 0.5                               # more than half of the readings missing
top_share = sensors[phase1].apply(lambda s: s.value_counts(normalize=True).max() if s.notna().any() else 1.0)
stuck_in_phase1 = (top_share > 0.95) & ~constant & ~mostly_empty     # one value in more than 95% of phase I
kept = ~constant & ~mostly_empty & ~stuck_in_phase1
S = sensors.loc[:, kept]
print(f"constant: {constant.sum()} | more than half empty: {mostly_empty.sum()} | "
      f"stuck on one value in phase I: {stuck_in_phase1.sum()} | kept: {kept.sum()}")

Now the labels, as a check. For each of the 432 sensors, a **rank test** (Mann-Whitney) asks: are the readings of the failed units higher or lower than those of the passed units? It works on ranks, so a few huge readings do not decide it. With 432 tests, chance alone gives about 22 below 0.05, so we use the false discovery rate, as in the drift lesson.

> **Predict.** How many sensors does BH keep over all the units? And inside one period?

> **Check.** You should see 23 over all the units, 2 in phase I only, and 0 in the comparison weeks only.

In [ ]:
def related_to_fails(mask):
    """For each sensor, the rank test's p-value: fails against passes, in the units of `mask`."""
    return pd.Series({c: stats.mannwhitneyu(S.loc[mask & fail, c].dropna(), S.loc[mask & ~fail, c].dropna()).pvalue
                      for c in S.columns})

p_all = related_to_fails(np.ones(len(fail), bool))
print(f"{len(p_all)} tests | p < 0.05: {(p_all < 0.05).sum()} (chance alone: about {0.05 * len(p_all):.0f}) | "
      f"Bonferroni: {(p_all < 0.05 / len(p_all)).sum()} | FDR (BH): {bh(p_all).sum()}")
related = list(p_all.index[bh(p_all)])
for name, m in [("phase I only", phase1), ("comparison weeks only", compare_weeks)]:
    print(f"{name}: FDR (BH) keeps {bh(related_to_fails(m)).sum()}")

Why does the number fall so much inside one period? Run the next cell. It tests which sensors moved between phase I and the rest, and the fail rate of both.

In [ ]:
ref = S[phase1]
p_time = pd.Series({c: stats.ks_2samp(ref[c].dropna(), S.loc[~phase1, c].dropna()).pvalue for c in S.columns})
moved = p_time.index[bh(p_time)]
print(f"sensors that moved between phase I and the rest (KS, FDR): {len(moved)} of {len(p_time)}")
print(f"of the {len(related)} related to fails, moved too: {len(set(related) & set(moved))}")
print(f"fail rate: phase I {fail[phase1].mean():.1%}, after it {fail[~phase1].mean():.1%}")

> **Check.** You should see that all 23 sensors related to fails also moved, and that the fail rate fell from 10.5% to 4.1%.

The sensors and the fail rate both changed with time. So a sensor that only follows the calendar looks "related to fails" over all the units. Inside one period, almost nothing is.

## 3. Phase I limits for every sensor

The I chart of [Control charts](https://learning.nextia-ai.com/courses/anomaly-detection/m03/control-charts/) takes its centre line from the mean of phase I and its sigma from the **moving range** (the mean change from one unit to the next, divided by 1.128). Run the next cell. It computes the limits of all 432 sensors and counts, for each unit, the sensors outside them.

> **Predict.** If every sensor were in control and bell-shaped, about 0.27% of the readings would be outside by chance: 1.17 sensors a unit. How many do you expect in phase I?

In [ ]:
centre = ref.mean()
sigma = ref.diff().abs().mean() / 1.128            # the I-MR chart's sigma, from the moving range
filled = S.fillna(ref.median())                    # a missing reading takes phase I's median: it gives no alarm
z = (filled - centre) / sigma                      # each reading, in sigmas from its centre line
beyond = z.abs() > 3                               # outside the I chart's limits
print(f"sensors outside their limits on one unit, phase I: {beyond[phase1].sum(axis=1).mean():.1f} on average "
      f"(chance alone, if every sensor were in control and bell-shaped: {0.0027 * S.shape[1]:.1f})")
print(f"the same in the comparison weeks: {beyond[compare_weeks].sum(axis=1).mean():.1f}")
print(f"units with no sensor outside: phase I {(beyond[phase1].sum(axis=1) == 0).sum()} of {phase1.sum()}")

> **Check.** You should see 12.11 in phase I and 15.3 in the comparison weeks.

Ten times more than chance, already in phase I. Real sensors have steps, drift and long tails, so phase I is not "in control" for most of them. Keep that in mind: an alarm on one sensor out of 432 means little.

## 4. The drift report for the first week

The drift report of [Drift in many features](https://learning.nextia-ai.com/courses/anomaly-detection/m03/drift-in-many-features/) compares the first week after phase I with phase I, sensor by sensor: the KS test, the PSI, and the BH decision. Run the next cell.

In [ ]:
def drift_report(mask):
    """One row per sensor: the KS p-value against phase I, the PSI, and the BH decision. Largest PSI first."""
    rep = pd.DataFrame({"p": [stats.ks_2samp(ref[c].dropna(), S.loc[mask, c].dropna()).pvalue for c in S.columns],
                        "PSI": [psi(ref[c], S.loc[mask, c]) for c in S.columns]}, index=S.columns)
    rep["BH"] = bh(rep.p)
    return rep.sort_values("PSI", ascending=False)

report = drift_report(first_week)
print(f"{len(report)} sensors | p < 0.05: {(report.p < 0.05).sum()} | FDR (BH): {report.BH.sum()} | "
      f"PSI 0.25 or more: {(report.PSI >= 0.25).sum()}")

> **Check.** You should see 246 sensors kept by BH and 205 with a PSI of 0.25 or more.

More than half of the sensors drifted in the first week. Now run the same report for every week after phase I, next to the fail rate of the week.

In [ ]:
weekly = pd.DataFrame([{"week": w, "units": int((week == w).sum()), "fails": int(fail[week == w].sum()),
                        "fail rate": round(fail[week == w].mean(), 3),
                        "sensors drifted (BH)": int(drift_report(week == w).BH.sum())}
                       for w in sorted(set(week[~phase1]))])
print(weekly.to_string(index=False))

> **Check.** In every week, BH keeps between 246 and 336 sensors. The week of 29 September has the most fails (15), but not many more drifted sensors than the others.

Drift is everywhere and all the time, and it does not follow the fails. A drift report on this line says "look", every week. It cannot say where the problem is.

## 5. Chart the sensors that moved most

A control chart only makes sense on a sensor that was in control in phase I. We call a sensor **chartable** if it has few missing readings, many different values, at most 1% of phase I outside its limits, and little memory from one unit to the next. From those, we take the 3 with the largest PSI in the first week's report. No label is used. Run the next cell.

In [ ]:
outside_phase1 = beyond[phase1].mean()                      # share of phase I units outside the limits
memory = ref.apply(lambda s: s.dropna().autocorr(1))        # how much a reading looks like the one before
chartable = ((S.isna().mean() < 0.05) & (ref.nunique() >= 50) & (outside_phase1 <= 0.01) & (memory.abs() < 0.3))
print(f"chartable sensors: {chartable.sum()} of {len(chartable)}")
chosen = list(report.index[report.BH & chartable[report.index]][:3])     # the 3 largest drifts that we can chart
print("chosen:", chosen)
print(report.loc[chosen].assign(p=lambda d: d.p.map("{:.1e}".format)).round(3).to_string())

> **Check.** You should see 151 chartable sensors and the chosen sensors `s059`, `s416`, `s051`.

Run the next cell. It draws the I chart of `s059`, with the phase I limits and every later unit.

In [ ]:
c = chosen[0]
x = np.arange(len(S))
fig, ax = plt.subplots()
ax.plot(x, filled[c], lw=0.7, color="#0054D8")
for v, style in [(centre[c], "-"), (centre[c] + 3 * sigma[c], "--"), (centre[c] - 3 * sigma[c], "--")]:
    ax.axhline(v, color="#4A5A76", ls=style, lw=1)
ax.axvline(phase1.sum(), color="#B42318", lw=1)
ax.set(title=f"I chart of {c}: phase I limits (left of the red line), then each new unit", xlabel="unit, in time order",
       ylabel=c)
plt.show()

The level of `s059` moved down after phase I and stayed there, but most single readings stay inside the limits. That is a small, lasting shift: the job of the EWMA. Run the next cell. It runs an EWMA on each chosen sensor (λ = 0.1, L = 2.6), and compares the fail rate while each EWMA is in alarm with the fail rate while it is not.

In [ ]:
lam, L = 0.1, 2.6                                    # the settings of the CUSUM and EWMA lesson
limit = L * np.sqrt(lam / (2 - lam))                 # the EWMA's limit, in sigmas, after the first days
start = np.nonzero(~phase1)[0][0]                    # each EWMA starts at 0 on the first unit after phase I
ew = pd.DataFrame({c: np.r_[ewma(z.loc[phase1, c].to_numpy()), ewma(z.loc[~phase1, c].to_numpy())] for c in chosen},
                  index=S.index)
alarm = ew.abs() > limit
rows = []
for c in chosen:
    first = labels.time[~phase1 & alarm[c].to_numpy()].min()
    rows.append({"sensor": c, "share in alarm, phase I": round(alarm.loc[phase1, c].mean(), 3),
                 "first alarm after phase I": f"{first:%d %b %H:%M}",
                 "share in alarm, comparison": round(alarm.loc[compare_weeks, c].mean(), 3),
                 "fail rate in alarm": round(fail[compare_weeks & alarm[c].to_numpy()].mean(), 3),
                 "fail rate not in alarm": round(fail[compare_weeks & ~alarm[c].to_numpy()].mean(), 3)})
ewma_table = pd.DataFrame(rows)
print(ewma_table.to_string(index=False))

> **Check.** You should see that the EWMA of `s059` is in alarm in 93% of the comparison units. Its fail rate in alarm is 4.8%, and not in alarm 7.5%.

The EWMA found the shift fast. But the units made during the alarm did not fail more often. The process moved; the quality did not get worse. This is what the labels are for: a check that the change matters.

## 6. One measure: fails found in 5 inspections a week

The quality team can take **5 units a week** out of the line for a full inspection (an assumption). Each detector gives every unit a score; each week, the team inspects the 5 highest. An inspection costs 80 dollars, and a fail that reaches the line test wastes 400 dollars of later work (both assumptions). The last column is the chance that random picks find as many fails.

> **Predict.** Which detector finds the most fails? Does any of them cost less than no inspection at all?

In [ ]:
BUDGET = 5                         # assumption: the quality team inspects 5 units a week, one each working day
INSPECT, MISSED = 80, 400          # assumptions, in dollars: one inspection; a fail that reaches the line test

def at_budget(score, mask=compare_weeks, budget=BUDGET):
    """Each week, inspect the `budget` units with the highest scores (ties: the earlier unit first)."""
    found = inspected = 0
    for w in sorted(set(week[mask])):
        units = np.nonzero(mask & (week == w))[0]
        top = units[np.argsort(-score[units], kind="stable")][:budget]
        inspected, found = inspected + len(top), found + int(fail[top].sum())
    weeks, fails = len(set(week[mask])), int(fail[mask].sum())
    luck = stats.binom.sf(found - 1, inspected, fail[mask].mean()) if inspected else 1.0
    return {"inspected": inspected, "fails found": found, "precision": round(found / max(inspected, 1), 3),
            "recall": round(found / fails, 3), "cost a week": round((INSPECT * inspected + MISSED * (fails - found)) / weeks),
            "chance of this by luck": round(luck, 3)}

mad = (filled[phase1] - ref.median()).abs().median() * 1.4826
forest = IsolationForest(n_estimators=200, random_state=34).fit(z[phase1])
scores = {
    "count outside limits (all sensors)": beyond.sum(axis=1).to_numpy(float),
    "count above 4 robust z (all sensors)": ((filled - ref.median()).abs() > 4 * mad.where(mad > 0)).sum(axis=1).to_numpy(float),
    "I chart, 3 chosen sensors (largest |z|)": z[chosen].abs().max(axis=1).to_numpy(),
    "EWMA, 3 chosen sensors (largest |EWMA|)": ew.abs().max(axis=1).to_numpy(),
    "Isolation Forest (all sensors)": -forest.score_samples(z),
}
rows = [{"detector": "no inspection", **at_budget(np.zeros(len(fail)), budget=0)}]
rows += [{"detector": name, **at_budget(s)} for name, s in scores.items()]
rows += [{"detector": "perfect (knows the answers)", **at_budget(fail.astype(float))}]
table = pd.DataFrame(rows)
print(f"comparison weeks: {len(set(week[compare_weeks]))} | units: {compare_weeks.sum()} | fails: {fail[compare_weeks].sum()} "
      f"| a random pick finds a fail {fail[compare_weeks].mean():.1%} of the time")
print(table.to_string(index=False))

> **Check.** You should see 3 fails found by `count outside limits (all sensors)`, a precision of 0.12, and a cost of 2,880 dollars a week for no inspection.

The simplest score, how many sensors are outside their limits, finds the most fails. The charts on the chosen sensors find 0 and 0: they watch the process, not the unit. But every number here is small: 3 fails in 25 inspections could happen by luck 13% of the time. An inspection pays only if more than 80 / 400 = 20% of the inspected units fail. No detector reaches that.

## 7. The end-to-end run on a new week

The last week, 13 to 17 October, was not used by any step. Run the weekly job on it: the drift report, the EWMA of the chosen sensors, and the 5 units to inspect, with the sensors that are furthest out on each one.

In [ ]:
def weekly_run(mask):
    """The weekly job on new units: the drift report, the EWMA of the chosen sensors, and the 5 units to inspect."""
    rep = drift_report(mask)
    print(f"{labels.time[mask].min():%d %b} to {labels.time[mask].max():%d %b}: {mask.sum()} units")
    print(f"drift against phase I: {rep.BH.sum()} sensors (BH); largest PSI:",
          ", ".join(f"{c} {v:.2f}" for c, v in rep.PSI.head(3).items()))
    print("EWMA of the chosen sensors, share of the week in alarm:", alarm.loc[mask, chosen].mean().round(2).to_dict())
    count = beyond.sum(axis=1)
    units = np.nonzero(mask)[0]
    top = units[np.argsort(-count.to_numpy()[units], kind="stable")][:BUDGET]
    card = pd.DataFrame({"time": labels.time.iloc[top].dt.strftime("%d %b %H:%M").to_numpy(),
                         "sensors outside": count.iloc[top].to_numpy(),
                         "furthest out": [", ".join(z.iloc[i].abs().nlargest(3).index) for i in top]},
                        index=pd.Index(top, name="unit"))
    return card, units

card, units = weekly_run(new_week)
print(card.to_string())

Now open the labels of the new week, as a check.

In [ ]:
count = beyond.sum(axis=1).to_numpy()
ranked = units[np.argsort(-count[units], kind="stable")]
failed = ranked[fail[ranked]]
print("fails among the 5 inspected:", int(fail[card.index].sum()), "| failed units this week:", [int(u) for u in failed],
      "| their rank by the count:", [int(np.nonzero(ranked == u)[0][0]) + 1 for u in failed], "of", len(units))

> **Check.** You should see `fails among the 5 inspected: 0 | failed units this week: [1519] | their rank by the count: [8] of 94`.

## 8. Check yourself

**Your turn 1.** Change one thing: the budget. What is the precision of `count outside limits (all sensors)` with 10 inspections a week? Use `at_budget()` with `budget=10`, and write it as `precision_10`.

In [ ]:
precision_10 = ...   # your code

In [ ]:
expect('precision_10', precision_10, 0.12, tolerance=1e-3)

**Your turn 2.** The week of 29 September had the most fails. How many sensors does BH keep in its drift report against phase I? The week is `week == "2008-09-29"`. Write the number as `drifted_sep29`.

In [ ]:
drifted_sep29 = ...   # your code

In [ ]:
expect('drifted_sep29', drifted_sep29, 331)

**Your turn 3.** In the new week, at which rank did the count put the unit that failed? Rank 1 is the highest count. Use `ranked` and `failed` from the last cell, and write the rank as `new_week_fail_rank`.

In [ ]:
new_week_fail_rank = ...   # your code

In [ ]:
expect('new_week_fail_rank', new_week_fail_rank, 8)

## Recap

- Of 590 sensors, 158 cannot carry a signal (constant, mostly empty or stuck). Of the other 432, the false discovery rate keeps 23 as related to fails over all the units, and almost none inside one period: most of the signal is time.
- Phase I is not in control for most real sensors: 12.11 sensors outside their limits on an average unit, against 1.17 by chance.
- The drift report keeps hundreds of sensors every week, and it does not follow the fails. The EWMA finds a lasting shift fast, and the labels show that the shift did not hurt the yield.
- Under one budget, the simplest score (sensors outside their limits) found the most fails, but no detector paid for its inspections.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Final assignment](https://learning.nextia-ai.com/courses/anomaly-detection/final-assignment/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/anomaly-detection/m10/control-charts-on-a-real-production-line/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The data is real, from a semiconductor line; it has no personal data.